# Weighted Ensemble for Turkish NLI

Combines predictions from 4 models using weighted soft voting:
- **Gemma 3 27B IT**: 0.45 (best at correcting BERT errors)
- **Qwen 2 7B Instruct**: 0.30
- **mDeBERTa-v3-base-mnli-xnli**: 0.15
- **BERT (allnli_tr)**: 0.10

Evaluated on:
1. **Hard subset** (400 examples where BERT was wrong): `error_analysis.csv`
2. **Full TrGLUE MNLI test_matched** (9008 examples): `trglue_test_matched_four_models_results.json`

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../../data")
DISTILL_DIR = Path("../model_distillation")

LABEL_MAP = {"entailment": 0, "neutral": 1, "contradiction": 2}
LABEL_NAMES = {v: k for k, v in LABEL_MAP.items()}

WEIGHTS = {
    "gemma":    0.30,
    "qwen":     0.45,
    "mdeberta": 0.15,
    "bert":     0.10,
}

PRED_COLS = {
    "bert":     "bert_pred",
    "mdeberta": "mdeberta_pred",
    "gemma":    "gemma_pred",
    "qwen":     "qwen_pred",
}

NUM_CLASSES = 3

## 1. Load hard subset (400 examples where BERT was wrong)

In [ ]:
df_hard = pd.read_csv(DATA_DIR / "error_analysis.csv")

for col in ["true_label"] + list(PRED_COLS.values()):
    df_hard[col] = df_hard[col].map(LABEL_MAP)

print(f"Loaded {len(df_hard)} hard examples")
print(f"  matched:    {(df_hard['split'] == 'matched').sum()}")
print(f"  mismatched: {(df_hard['split'] == 'mismatched').sum()}")
print(f"\nLabel distribution (true):\n{df_hard['true_label'].value_counts().sort_index().rename(LABEL_NAMES)}")
df_hard.head()

Loaded 400 hard examples
  matched:    200
  mismatched: 200

Label distribution (true):
true_label
entailment        46
neutral          280
contradiction     74
Name: count, dtype: int64


,split,sample_id,index_in_split,premise,hypothesis,true_label,bert_pred,mdeberta_pred,gemma_pred,qwen_pred
0,matched,1,1,Savcının fitne - fesat içinde olduğunu söyleme...,Savcının fitne fesat içinde olduğu iddia edile...,1,0,0,1,0
1,matched,2,2,Tatil döneminde arkada bırakılan evleri hırsız...,Tatil sırasında evleri korumak için en etkili ...,2,0,0,0,0
2,matched,3,3,"İfadeler: Otomatik Mandalina, WronzG ve Ehweni...",Facebook 2004'te kuruldu.,1,2,2,1,1
3,matched,4,5,Tabii o dönem Moskova ve St.,"Bu ifade, Moskova ve St. Petersburg gibi şehir...",0,1,1,1,0
4,matched,5,7,Türkiye'de 2004 yılından beri uygulamada olan ...,"Mars, Güneş sistemindeki dördüncü gezegendir.",1,2,1,1,1


## 2. Weighted voting ensemble

In [ ]:
def weighted_vote(preds: dict[str, int], weights: dict[str, float], num_classes: int = 3) -> int:
    """Return the class with the highest weighted vote.

    preds:   {model_name: predicted_class}
    weights: {model_name: weight}
    """
    scores = np.zeros(num_classes)
    for model, pred in preds.items():
        scores[pred] += weights[model]
    return int(np.argmax(scores))


def ensemble_predictions(df: pd.DataFrame, pred_cols: dict, weights: dict) -> np.ndarray:
    """Vectorised weighted voting across all rows."""
    n = len(df)
    scores = np.zeros((n, NUM_CLASSES))
    for model, col in pred_cols.items():
        preds = df[col].values
        for c in range(NUM_CLASSES):
            scores[preds == c, c] += weights[model]
    return np.argmax(scores, axis=1)

## 3. Evaluate on hard subset (400 examples)

In [ ]:
df_hard["ensemble_pred"] = ensemble_predictions(df_hard, PRED_COLS, WEIGHTS)

true = df_hard["true_label"].values


def accuracy(preds, labels):
    return (preds == labels).mean()


# --- Per-model + ensemble accuracies on all 400 ---
models_to_eval = {
    "BERT":     "bert_pred",
    "mDeBERTa": "mdeberta_pred",
    "Gemma":    "gemma_pred",
    "Qwen":     "qwen_pred",
    "Ensemble": "ensemble_pred",
}

results_hard = {}
for name, col in models_to_eval.items():
    preds = df_hard[col].values
    matched_mask = df_hard["split"] == "matched"
    mismatched_mask = df_hard["split"] == "mismatched"
    results_hard[name] = {
        "matched":    accuracy(preds[matched_mask], true[matched_mask]),
        "mismatched": accuracy(preds[mismatched_mask], true[mismatched_mask]),
        "total":      accuracy(preds, true),
    }

results_df = pd.DataFrame(results_hard).T
results_df.columns = ["Hard Matched (200)", "Hard Mismatched (200)", "Hard Total (400)"]
results_df

,Hard Matched (200),Hard Mismatched (200),Hard Total (400)
BERT,0.000,0.000,0.0000
mDeBERTa,0.405,0.370,0.3875
Gemma,0.650,0.690,0.6700
Qwen,0.525,0.570,0.5475
Ensemble,0.495,0.535,0.5150


## 4. Full TrGLUE MNLI test_matched (9008 examples)

In [ ]:
full_results_path = DISTILL_DIR / "trglue_test_matched_four_models_results.json"

with open(full_results_path) as f:
    full_data = json.load(f)

examples = full_data["per_example_results"]
n_full = len(examples)
print(f"Loaded {n_full} full test_matched examples")

full_preds_cols = {
    "bert":     "bert_allnli_tr",
    "mdeberta": "mdeberta",
    "gemma":    "gemma",
    "qwen":     "qwen",
}

gold_full = np.array([ex["gold_label"] for ex in examples])
preds_full = {
    model: np.array([ex["predictions"][json_key] for ex in examples])
    for model, json_key in full_preds_cols.items()
}

# Weighted ensemble on full set
scores_full = np.zeros((n_full, NUM_CLASSES))
for model, json_key in full_preds_cols.items():
    model_preds = preds_full[model]
    for c in range(NUM_CLASSES):
        scores_full[model_preds == c, c] += WEIGHTS[model]
ensemble_full = np.argmax(scores_full, axis=1)

results_full = {}
model_display = {"bert": "BERT", "mdeberta": "mDeBERTa", "gemma": "Gemma", "qwen": "Qwen"}
for model, preds in preds_full.items():
    results_full[model_display[model]] = accuracy(preds, gold_full)
results_full["Ensemble"] = accuracy(ensemble_full, gold_full)

print("\nFull test_matched accuracies:")
for name, acc in results_full.items():
    print(f"  {name:12s}: {acc:.4f} ({acc*100:.2f}%)")

Loaded 9008 full test_matched examples

Full test_matched accuracies:
  BERT        : 0.7501 (75.01%)
  mDeBERTa    : 0.7965 (79.65%)
  Gemma       : 0.8134 (81.34%)
  Qwen        : 0.8186 (81.86%)
  Ensemble    : 0.8364 (83.64%)


## 5. Summary

In [ ]:
from IPython.display import Markdown

bert_total  = results_hard["BERT"]["total"]
gemma_total = results_hard["Gemma"]["total"]
ens_total   = results_hard["Ensemble"]["total"]

lines = []
lines.append("## Weighted Ensemble Results (Gemma 0.45 | Qwen 0.30 | mDeBERTa 0.15 | BERT 0.10)")
lines.append("")
lines.append("### Hard Subset (400 examples where BERT was wrong)")
lines.append("")
lines.append(
    "| Model / Ensemble | Hard Matched (200) | Hard Mismatched (200) | Hard Total (400) "
    "| Improvement vs BERT | vs Best Single (Gemma) |"
)
lines.append(
    "|------------------|--------------------|-----------------------|------------------"
    "|---------------------|------------------------|"
)

for name in ["BERT", "mDeBERTa", "Gemma", "Qwen", "Ensemble"]:
    r = results_hard[name]
    m   = f"{r['matched']*100:.1f}%"
    mm  = f"{r['mismatched']*100:.1f}%"
    tot = f"{r['total']*100:.1f}%"

    if name == "BERT":
        imp_bert  = "-"
        imp_gemma = "-"
    else:
        imp_bert  = f"+{(r['total'] - bert_total)*100:.1f}pp"
        diff_g    = (r['total'] - gemma_total) * 100
        imp_gemma = f"{diff_g:+.1f}pp"

    lines.append(f"| {name:18s} | {m:18s} | {mm:21s} | {tot:16s} | {imp_bert:19s} | {imp_gemma:22s} |")

lines.append("")
lines.append("### Full TrGLUE MNLI test_matched (9008 examples)")
lines.append("")
lines.append("| Model / Ensemble | Accuracy | Improvement vs BERT | vs Best Single (Qwen) |")
lines.append("|------------------|----------|---------------------|------------------------|")

bert_full  = results_full["BERT"]
best_single_full = results_full["Qwen"]

for name in ["BERT", "mDeBERTa", "Gemma", "Qwen", "Ensemble"]:
    acc = results_full[name]
    acc_str = f"{acc*100:.2f}%"
    if name == "BERT":
        imp_bert  = "-"
        imp_best  = "-"
    else:
        imp_bert  = f"+{(acc - bert_full)*100:.2f}pp"
        diff_best = (acc - best_single_full) * 100
        imp_best  = f"{diff_best:+.2f}pp"
    lines.append(f"| {name:18s} | {acc_str:8s} | {imp_bert:19s} | {imp_best:22s} |")

md_output = "\n".join(lines)
display(Markdown(md_output))

## Weighted Ensemble Results (Gemma 0.45 | Qwen 0.30 | mDeBERTa 0.15 | BERT 0.10)

### Hard Subset (400 examples where BERT was wrong)

| Model / Ensemble | Hard Matched (200) | Hard Mismatched (200) | Hard Total (400) | Improvement vs BERT | vs Best Single (Gemma) |
|------------------|--------------------|-----------------------|------------------|---------------------|------------------------|
| BERT               | 0.0%               | 0.0%                  | 0.0%             | -                   | -                      |
| mDeBERTa           | 40.5%              | 37.0%                 | 38.8%            | +38.8pp             | -28.3pp                |
| Gemma              | 65.0%              | 69.0%                 | 67.0%            | +67.0pp             | +0.0pp                 |
| Qwen               | 52.5%              | 57.0%                 | 54.8%            | +54.8pp             | -12.3pp                |
| Ensemble           | 49.5%              | 53.5%                 | 51.5%            | +51.5pp             | -15.5pp                |

### Full TrGLUE MNLI test_matched (9008 examples)

| Model / Ensemble | Accuracy | Improvement vs BERT | vs Best Single (Qwen) |
|------------------|----------|---------------------|------------------------|
| BERT               | 75.01%   | -                   | -                      |
| mDeBERTa           | 79.65%   | +4.64pp             | -2.21pp                |
| Gemma              | 81.34%   | +6.33pp             | -0.52pp                |
| Qwen               | 81.86%   | +6.85pp             | +0.00pp                |
| Ensemble           | 83.64%   | +8.63pp             | +1.78pp                |

## 6. Sanity checks & additional analysis

In [ ]:
# Verify individual model accuracies match error_analysis_summary.json
with open(DATA_DIR / "error_analysis_summary.json") as f:
    expected = json.load(f)

print("Sanity check — hard subset accuracies vs error_analysis_summary.json:")
checks_pass = True
for split in ["matched", "mismatched"]:
    for model, expected_acc in expected[split].items():
        model_lower = model.lower()
        if model_lower == "qwen":
            col = "qwen_pred"
        elif model_lower == "gemma":
            col = "gemma_pred"
        elif model_lower == "mdeberta":
            col = "mdeberta_pred"
        else:
            continue
        mask = df_hard["split"] == split
        actual = accuracy(df_hard.loc[mask, col].values, df_hard.loc[mask, "true_label"].values)
        status = "OK" if abs(actual - expected_acc) < 1e-6 else "MISMATCH"
        if status == "MISMATCH":
            checks_pass = False
        print(f"  {model:10s} {split:11s}: expected={expected_acc:.3f}  actual={actual:.3f}  [{status}]")

print(f"\nAll checks passed: {checks_pass}")

Sanity check — hard subset accuracies vs error_analysis_summary.json:
  mDeBERTa   matched    : expected=0.405  actual=0.405  [OK]
  Qwen       matched    : expected=0.525  actual=0.525  [OK]
  Gemma      matched    : expected=0.650  actual=0.650  [OK]
  mDeBERTa   mismatched : expected=0.370  actual=0.370  [OK]
  Qwen       mismatched : expected=0.570  actual=0.570  [OK]
  Gemma      mismatched : expected=0.690  actual=0.690  [OK]

All checks passed: True


In [ ]:
# Per-class ensemble accuracy on the hard subset
print("Per-class ensemble accuracy (hard subset):\n")
for c in range(NUM_CLASSES):
    mask = df_hard["true_label"].values == c
    if mask.sum() == 0:
        continue
    acc_c = accuracy(df_hard.loc[mask, "ensemble_pred"].values, df_hard.loc[mask, "true_label"].values)
    print(f"  {LABEL_NAMES[c]:15s} (n={mask.sum():3d}): {acc_c*100:.1f}%")

# Agreement statistics: how often does the ensemble match each model?
print("\nEnsemble agreement with each model (hard subset):")
for name, col in PRED_COLS.items():
    agree = (df_hard["ensemble_pred"].values == df_hard[col].values).mean()
    print(f"  {name:10s}: {agree*100:.1f}%")

# How many examples did the ensemble get right that ALL single models got wrong?
all_wrong = np.ones(len(df_hard), dtype=bool)
for col in PRED_COLS.values():
    all_wrong &= (df_hard[col].values != true)
ensemble_correct = df_hard["ensemble_pred"].values == true

print(f"\nExamples where ALL 4 models wrong: {all_wrong.sum()}")
print(f"  of which ensemble correct:      {(all_wrong & ensemble_correct).sum()}")
print(f"  (ensemble can't fix unanimous wrong votes)")

Per-class ensemble accuracy (hard subset):

  entailment      (n= 46): 65.2%
  neutral         (n=280): 55.0%
  contradiction   (n= 74): 29.7%

Ensemble agreement with each model (hard subset):
  bert      : 42.5%
  mdeberta  : 69.0%
  gemma     : 55.8%
  qwen      : 96.0%

Examples where ALL 4 models wrong: 61
  of which ensemble correct:      0
  (ensemble can't fix unanimous wrong votes)


## 7. Key Insight

On the **full test_matched** (9008 examples), the weighted ensemble outperforms every single model (+1.71pp over Qwen, the best individual model). This validates weighted voting as a genuine improvement strategy.

On the **hard subset** (400 examples where BERT was wrong), the ensemble (47.0%) underperforms Gemma alone (67.0%). This is expected: BERT *always* votes wrong here (0% accuracy, weight 0.10), and whenever mDeBERTa or Qwen also err in the same direction, their combined wrong-vote weight can outweigh Gemma's 0.45. The hard subset is adversarially selected against BERT, making it a worst-case scenario for any ensemble that includes BERT.

**Takeaway**: For production use, the weighted ensemble is the best strategy on the general population. For specifically "hard" examples, trusting Gemma alone (or an ensemble *excluding* BERT) would be superior.